# Solar V4 complete reproduction

This notebook is inert by default. Use ordinary Kaggle T4; configure/install the
public pinned offline wheel before mounting data in an offline GPU session.
The explicit checkout path below follows the README. Change it if your checkout
is elsewhere. Source and fixed splits are read from that repository root.
No private v3 output or original local workspace is needed.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

PROJECT_ROOT = Path("/kaggle/working/solar-filament-2026-reproducible")
DATA_ROOT = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
NEW_OUTPUT = PROJECT_ROOT / "runs/notebook_v4_reproduction"
RUN = False
RUN_INFERENCE = False
assert (PROJECT_ROOT / "src/solarseg/v4_runner.py").is_file()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
configuration = json.loads((PROJECT_ROOT / "configs/v4_experiments.json").read_text())
splits = json.loads((PROJECT_ROOT / "configs/v4_splits.json").read_text())
print(json.dumps(configuration, indent=2))
print("Exact split SHA256:", hashlib.sha256((PROJECT_ROOT / "configs/v4_splits.json").read_bytes()).hexdigest())


## Official data and supervision

Obtain competition data through your own Kaggle access after personally accepting
its rules. Do not download full external MAGFiLO annotations or any test labels.
The native cache is built only for the exact development IDs in the fixed split
file. Each annotator's instance polygons form a union, then annotators are averaged
for the semantic training target. Official PQ still compares annotators separately.
The previously observed 2020–2022 holdout is excluded and is not evaluated again.


In [ ]:
print("Development images:", len(splits["development_ids"]))
print("Development folds:", configuration["folds"])
print("Public v3 count reference:", PROJECT_ROOT / "evidence/v3")


## Prespecified training, sliding-window inference and evaluation

The runner constructs native 2048² raw/limb-corrected images, radius channels and
consensus targets; trains both fixed arms on native 512² crops with matched
foreground/disk sampling; blends 25 overlapping tiles before instance aggregation;
and evaluates each frozen aggregation rule once. Period/year tables derive from
the same per-image counts. It saves sample draws, losses, probability diagnostics,
predictions, PQ/TP/FP/FN, runtime and GPU memory, plus failure cases.
Set `RUN=True` only to authorize this heavy Kaggle workload. Existing output
directories are not overwritten. No score from the leaderboard enters this run.


In [ ]:
if RUN:
    environment = dict(os.environ)
    environment.update(PYTHONPATH=str(PROJECT_ROOT / "src"), CUBLAS_WORKSPACE_CONFIG=":4096:8",
                       OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="4")
    subprocess.run([sys.executable, "-u", "-m", "solarseg.v4_runner",
        "--data-root", str(DATA_ROOT), "--v3-root", str(PROJECT_ROOT / "evidence/v3"),
        "--config", str(PROJECT_ROOT / "configs/v4_experiments.json"),
        "--output", str(NEW_OUTPUT)], cwd=PROJECT_ROOT, env=environment, check=True)


## Frozen selected prediction without retraining

The released selected v3 method is classical; its parameters in
`models/final_v1/model.json` suffice, with no learned weight download.
Set `RUN_INFERENCE=True` to reproduce the selected predictions into a new file.
The CSV has one nonempty instance per row and compressed 2048² COCO RLE. Natural
zero-instance images receive no dummy rows. Kaggle accepted the frozen original
as submission 57795106 with Public Score 0.15; that fact does not tune a threshold.


In [ ]:
if RUN_INFERENCE:
    environment = dict(os.environ, PYTHONPATH=str(PROJECT_ROOT / "src"))
    subprocess.run([sys.executable, "-u", "-m", "solarseg.cli", "infer",
        "--data-root", str(DATA_ROOT), "--model", str(PROJECT_ROOT / "models/final_v1/model.json"),
        "--device", "cuda", "--output", str(PROJECT_ROOT / "submissions/notebook_reproduced_v3.csv")],
        cwd=PROJECT_ROOT, env=environment, check=True)


## Evidence and final entry

The frozen v3 official record is `docs/official_submission_v3.md`; counts and exact
manifest are under `evidence/v3/`. The P1 audit and P2 diagnosis document known
failure mechanisms and numeric conventions. Preserve full generated V4 runtime
requirements and model weights when publishing any newly selected neural model.
Use the four-page report sources and `report/google_form_checklist.md` for final
entry. No cell submits a competition prediction or Google Form.
